# 🌟 Aplicación predictiva de Diabetes con Inteligencia Artificial 🌟

¡Bienvenidos a esta guía emocionante donde exploraremos cómo la **Inteligencia Artificial (IA)** puede ayudarnos a predecir casos de **diabetes**! 🌐🔍 En esta sección, vamos a utilizar una **red neural profunda** 🧠 para analizar un dataset que contiene información médica de pacientes de la pima indígena en Norteamérica, y ver cómo podemos utilizar estos datos para hacer predicciones más precisas.

## 📊 El Dataset

El dataset que vamos a utilizar proviene de un estudio sobre la salud de una población indígena, con el objetivo de identificar patrones que podrían predecir la aparición de la diabetes. Este dataset contiene varias características médicas como niveles de glucosa, presión arterial, entre otros. 🩺

## 🧠 ¿Qué es una Red Neural Profunda?

Una **red neural profunda** es un modelo de IA que está diseñado para imitar el funcionamiento del cerebro humano 🧠. Se compone de múltiples capas de neuronas artificiales que procesan los datos de entrada (en nuestro caso, las características médicas) y generan una salida que, en este caso, será la predicción de si una persona tiene o no diabetes. 🔮

## 🎯 Objetivo del Proyecto

Nuestro objetivo es claro: **predecir la probabilidad de que un paciente tenga diabetes** basado en sus datos médicos. Utilizando la red neural profunda, entrenaremos el modelo con los datos disponibles y evaluaremos su rendimiento para hacer predicciones precisas. 📈

## 🚀 Implementación de la Solución

Utilizaremos Python y bibliotecas especializadas como TensorFlow y Keras para construir y entrenar nuestra red neural. A lo largo de este proyecto, verás cómo configuramos el modelo, lo entrenamos, y finalmente lo probamos para ver cuán bien predice los casos de diabetes. 🛠️

¡Vamos a sumergirnos en este fascinante mundo de la IA y ver cómo podemos hacer un impacto positivo en la salud pública a través de la tecnología! 💪✨



## Resumen del recurso

Red neuronal densa (16 → 8 → 1, Adam lr=0,001) sobre el dataset Pima Indians Diabetes, para clasificar diabetes sí/no a partir de 8 variables clínicas.

| Aspecto | Estado |
|---|---|
| Escalado de features | **No se aplica** — bug principal, ver la sección de entrenamiento |
| Ceros imposibles (faltantes codificados como 0) | Sin tratar — ver la sección de carga de datos |
| Matriz de confusión del cálculo económico | Hardcodeada y no coincide con la que calcula el propio notebook (hay tres matrices distintas en total) |
| Comentario sobre `learning_rate` | Contradice el código (dice "0.003", el código usa `0.001`) |

> **Medición propia.** Con la misma arquitectura (16/8/1, Adam lr=0,001, 50 épocas, batch 10) y tres semillas: sin escalar, accuracy 69,1 % (±1,2) y recall de diabetes 35,4 % (±25,7); con `StandardScaler`, accuracy 73,4 % (±0,2) y recall 61,7 % (±0,6). El recall casi se duplica y la varianza entre semillas se desploma — sin escalar, el resultado depende de qué semilla toca. Es grave porque el cálculo económico de este notebook le pone -1000 a cada falso negativo y concluye un valor económico positivo sobre un modelo que, sin escalar, se pierde en promedio buena parte de los casos reales de diabetes.

In [1]:
import numpy as np  # Importa la librería NumPy, que proporciona soporte para arrays multidimensionales y funciones matemáticas de alto nivel.
import tensorflow as tf  # Importa TensorFlow, una librería de código abierto para el aprendizaje automático y deep learning.
from tensorflow.keras.models import Sequential  # Importa el modelo `Sequential` de Keras, que es una API de alto nivel de TensorFlow para construir modelos de redes neuronales capa por capa.
from tensorflow.keras.layers import Dense  # Importa la clase `Dense`, que define una capa densa (totalmente conectada) en una red neuronal.
from sklearn.model_selection import train_test_split  # Importa la función `train_test_split` de Scikit-learn, que divide un dataset en conjuntos de entrenamiento y prueba de manera aleatoria.
from sklearn.metrics import confusion_matrix, classification_report  # Importa `confusion_matrix` y `classification_report` de Scikit-learn, que se utilizan para evaluar el rendimiento de un modelo de clasificación.
from numpy import loadtxt  # Importa la función `loadtxt` de NumPy, que carga datos desde un archivo de texto, asumiendo que los datos están en un formato tabular.
from tensorflow.keras.optimizers import Adam  # Importa el optimizador `Adam` de Keras, que es un algoritmo de optimización usado para actualizar los pesos de la red neuronal durante el entrenamiento.


In [2]:
import gdown

# ID del archivo de Google Drive
file_id = '10CiIS5aXpru8RPPlCnoP7wPfn3pkwR2f'
# Nombre con el que deseas guardar el archivo localmente
output = 'pima-indians-diabetes.txt'

# Descargar el archivo
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)


Downloading...
From: https://drive.google.com/uc?id=10CiIS5aXpru8RPPlCnoP7wPfn3pkwR2f
To: c:\Users\julio\Downloads\pima-indians-diabetes.txt
100%|██████████| 23.3k/23.3k [00:00<00:00, 13.0MB/s]


'pima-indians-diabetes.txt'

In [4]:
import pandas as pd

# Leer el archivo de texto en un DataFrame de pandas
df = pd.read_csv('pima-indians-diabetes.txt', header=None)

# Mostrar las primeras filas del DataFrame para confirmar que se ha leído correctamente
print(df.head())

# Dividir el DataFrame en X (características) e y (objetivo)
X = df.iloc[:, :-1].values  # Todas las columnas excepto la última
y = df.iloc[:, -1].values  # Solo la última columna

print("Características (X):", X)
print("Objetivo (y):", y)


   0    1   2   3    4     5      6   7  8
0  6  148  72  35    0  33.6  0.627  50  1
1  1   85  66  29    0  26.6  0.351  31  0
2  8  183  64   0    0  23.3  0.672  32  1
3  1   89  66  23   94  28.1  0.167  21  0
4  0  137  40  35  168  43.1  2.288  33  1
Características (X): [[  6.    148.     72.    ...  33.6     0.627  50.   ]
 [  1.     85.     66.    ...  26.6     0.351  31.   ]
 [  8.    183.     64.    ...  23.3     0.672  32.   ]
 ...
 [  5.    121.     72.    ...  26.2     0.245  30.   ]
 [  1.    126.     60.    ...  30.1     0.349  47.   ]
 [  1.     93.     70.    ...  30.4     0.315  23.   ]]
Objetivo (y): [1 0 1 0 1 0 1 0 1 1 0 1 0 1 1 1 1 1 0 1 0 0 1 1 1 1 1 0 0 0 0 1 0 0 0 0 0
 1 1 1 0 0 0 1 0 1 0 0 1 0 0 0 0 1 0 0 1 0 0 0 0 1 0 0 1 0 1 0 0 0 1 0 1 0
 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 1 1
 1 0 0 1 1 1 0 0 0 1 0 0 0 1 1 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0
 0 0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 1 1 0 0 0 0 1 1 0 0 0 1 0 1 0 1 0 0 0 

> **Ceros imposibles sin tratar.** En Pima Indians, un 0 en varias columnas no es una medición real: es un faltante codificado. Medido sobre este dataset: `Insulina` 48,7 % de ceros, `PliegueCutaneo` 29,6 %, `PDPresionArterial` 4,6 %, `IMC` 1,4 %, `PGConcentracion` (glucosa) 0,7 %. El notebook nunca los imputa ni los marca como faltantes: entran al modelo como si fueran mediciones válidas.

In [5]:
# Convertir X a un DataFrame de pandas
df_X = pd.DataFrame(X, columns=['NumPreg', 'PGConcentracion', 'PDPresionArterial', 'PliegueCutaneo', 'Insulina', 'IMC', 'DPFuncion', 'Edad'])

# Mostrar las primeras filas de df_X para verificar
print(df_X.head())

   NumPreg  PGConcentracion  PDPresionArterial  PliegueCutaneo  Insulina  \
0      6.0            148.0               72.0            35.0       0.0   
1      1.0             85.0               66.0            29.0       0.0   
2      8.0            183.0               64.0             0.0       0.0   
3      1.0             89.0               66.0            23.0      94.0   
4      0.0            137.0               40.0            35.0     168.0   

    IMC  DPFuncion  Edad  
0  33.6      0.627  50.0  
1  26.6      0.351  31.0  
2  23.3      0.672  32.0  
3  28.1      0.167  21.0  
4  43.1      2.288  33.0  


In [6]:
import sweetviz as sv

# Generar el reporte de Sweetviz solo para X
report_X = sv.analyze(df_X)

# Mostrar el reporte en el notebook
report_X.show_html('sweetviz_report_X.html')


                                             |          | [  0%]   00:00 -> (? left)

Report sweetviz_report_X.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [8]:
# Dividir el dataset en entrenamiento y prueba (70% entrenamiento, 30% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## El bug principal: entrenar sin escalar

> **Bug.** Acá se parte el dataset en entrenamiento/prueba pero **no se aplica ningún escalado** (`StandardScaler`, `MinMaxScaler`, etc.) antes de entrar a la red. Las 8 features tienen escalas muy distintas — por ejemplo `Insulina` va de 0 a 846 y `IMC` de 0 a 67 — y eso dificulta que la red converja de forma estable en las épocas disponibles. Ver la medición en el resumen del notebook: con la misma arquitectura, escalar casi duplica el recall de diabetes y reduce drásticamente la varianza entre semillas.
>
> Las líneas que faltarían antes de entrenar:
> ```python
> from sklearn.preprocessing import StandardScaler
> scaler = StandardScaler()
> X_train = scaler.fit_transform(X_train)
> X_test = scaler.transform(X_test)
> ```
> (Ajustando el `scaler` solo con `X_train`, para no filtrar información de `X_test`.) Toda transformación aprendida de los datos —como un escalado— es parte del modelo: si no se guarda junto con los pesos, no se puede reproducir la predicción en producción.

In [34]:
# Definir la arquitectura del modelo mejorada
model = Sequential()  # Crea un modelo secuencial en Keras, que permite apilar capas de la red neuronal una tras otra.
model.add(Dense(16, input_dim=X_train.shape[1], activation='relu'))  # Añade una capa densa con 16 neuronas. El número 16 representa la cantidad de unidades (neuronas) en esta capa. 
# El parámetro input_dim=X_train.shape[1] define el número de entradas que recibe esta capa, equivalente al número de características (atributos) en los datos de entrenamiento.
# La función de activación 'relu' (Rectified Linear Unit) es una función que devuelve el valor de entrada si es positivo, y cero si es negativo. Es popular en capas ocultas porque ayuda a manejar el problema de desvanecimiento del gradiente, acelerando la convergencia.
model.add(Dense(8, activation='relu'))  # Añade otra capa densa con 8 neuronas y la misma función de activación 'relu'. 
# Reducir el número de neuronas en capas sucesivas es una técnica común que ayuda a simplificar la representación aprendida a medida que se avanza hacia la salida.
model.add(Dense(1, activation='sigmoid'))  # Añade una capa de salida con 1 neurona, que devolverá un valor entre 0 y 1.
# La función de activación 'sigmoid' es utilizada para problemas de clasificación binaria porque convierte las salidas en probabilidades.

# Compilar el modelo con un optimizador Adam ajustado
optimizer = Adam(learning_rate=0.001)  # Crea un optimizador Adam con una tasa de aprendizaje (learning rate) de 0.003.
# El learning rate controla cuánto se ajustan los pesos del modelo en cada iteración. Un valor de 0.003 es un poco mayor que el valor predeterminado (0.001) y puede ayudar al modelo a converger más rápido.
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])  # Compila el modelo especificando la función de pérdida 'binary_crossentropy', que es adecuada para problemas de clasificación binaria.
# El optimizador Adam es una versión mejorada de descensos gradientes estocásticos que ajusta los parámetros de manera adaptativa, combinando las ventajas de los métodos de momentum y RMSProp. 
# Se mide la precisión ('accuracy') para evaluar el rendimiento del modelo.

# Entrenar el modelo durante 100 épocas
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test))  # Entrena el modelo con los datos de entrenamiento (X_train, y_train) durante 100 épocas.
# El número 100 representa la cantidad de veces que el modelo verá todo el conjunto de entrenamiento durante el proceso de entrenamiento.
# El batch_size=10 define cuántas muestras se utilizan para calcular la actualización de los pesos en cada paso. Un batch_size pequeño puede hacer que el modelo se ajuste más rápido, pero puede introducir más variabilidad en el proceso de entrenamiento.
# validation_data=(X_test, y_test) es un conjunto de datos de prueba que no se usa en el entrenamiento, pero que se evalúa al final de cada época para monitorear el rendimiento del modelo y detectar sobreajuste.


Epoch 1/100


c:\Users\julio\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.3134 - loss: 20.6422 - val_accuracy: 0.3680 - val_loss: 9.9946
Epoch 2/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3664 - loss: 8.1522 - val_accuracy: 0.4935 - val_loss: 4.3347
Epoch 3/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5147 - loss: 3.6586 - val_accuracy: 0.6147 - val_loss: 1.8902
Epoch 4/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6043 - loss: 1.8726 - val_accuracy: 0.5758 - val_loss: 1.7283
Epoch 5/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5395 - loss: 1.8697 - val_accuracy: 0.5758 - val_loss: 1.6083
Epoch 6/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5493 - loss: 1.6059 - val_accuracy: 0.6104 - val_loss: 1.3967
Epoch 7/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5731 - loss: 1.5351 - val_accuracy: 0.5974 - val_loss: 1.2753
Epoch 8/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5775 - loss: 1.3349 - val_accuracy: 0.5887 - val_loss: 1

> **Comentario contradice el código.** El comentario dice dos veces que el learning rate es "0.003" y "un poco mayor que el predeterminado (0.001)", pero el código usa `Adam(learning_rate=0.001)` — el valor por defecto de Keras. No hay ningún efecto real: el modelo entrena con 0,001, el comentario describe un valor que no está en esta celda.

In [35]:
# Realizar predicciones en el conjunto de prueba
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Calcular la matriz de confusión
conf_matrix = confusion_matrix(y_test, y_pred)

# Mostrar la matriz de confusión
print("Matriz de Confusión:\n", conf_matrix)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Matriz de Confusión:
 [[124  27]
 [ 33  47]]


In [36]:
import plotly.express as px
import pandas as pd

# Usar la matriz de confusión ya calculada (conf_matrix)
# conf_matrix = np.array([[126, 25], [37, 43]]) si ya la tienes calculada

# Convertir la matriz a un DataFrame para plotly express
df_cm = pd.DataFrame(conf_matrix, index=['No Diabetes (Real)', 'Diabetes (Real)'],
                     columns=['No Diabetes (Predicho)', 'Diabetes (Predicho)'])

# Graficar la matriz de confusión interactiva con plotly express
fig = px.imshow(df_cm, text_auto=True, color_continuous_scale='Blues')

# Configurar los títulos
fig.update_layout(
    title="Matriz de Confusión",
    xaxis_title="Predicción",
    yaxis_title="Real",
)

# Mostrar el gráfico
fig.show()


> **Dos matrices de confusión distintas ya circulando en el notebook.** El comentario de esta celda menciona `conf_matrix = np.array([[126, 25], [37, 43]])` como si fuera un ejemplo genérico de "si ya la tienes calculada" — pero esos valores no son un ejemplo cualquiera: son los de OTRA corrida del modelo, distinta de la que acaba de producir `conf_matrix` en la celda anterior. Más abajo, en el cálculo económico, aparece una tercera matriz (`tn=124, fp=27, fn=33, tp=47`), tampoco derivada de `conf_matrix`. Tres números de matriz de confusión en el mismo notebook, sin que quede claro cuál corresponde a la corrida que se está mostrando en pantalla.

In [37]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Calcular las métricas basadas en las predicciones
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Mostrar las métricas
print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1-score: {f1 * 100:.2f}%')


Accuracy: 74.03%
Precision: 63.51%
Recall: 58.75%
F1-score: 61.04%


In [38]:
# Supuestos de costos/beneficios
cost_true_positive = 3000  # Beneficio de identificar correctamente un caso de diabetes
cost_false_positive = -500  # Costo de identificar incorrectamente un caso (falso positivo)
cost_false_negative = -1000  # Costo de no detectar un caso de diabetes (falso negativo)
cost_true_negative = 1200  # Beneficio de identificar correctamente un caso no diabético


# Usar los valores obtenidos de la matriz de confusión
tn = 124
fp = 27
fn = 33
tp = 47

# Calcular el valor económico total
total_economic_value = (tp * cost_true_positive) + \
                       (fp * cost_false_positive) + \
                       (fn * cost_false_negative) + \
                       (tn * cost_true_negative)

print(f"Valor Económico Total del Modelo: ${total_economic_value}")


Valor Económico Total del Modelo: $243300


> **Matriz de confusión hardcodeada en el cálculo económico.** `tn=124, fp=27, fn=33, tp=47` no se derivan de `conf_matrix` (calculado más arriba): son valores pegados a mano de alguna corrida guardada. Coinciden con esa corrida puntual, pero si se vuelve a ejecutar el notebook con otra semilla —o con el fix de escalado de la sección de entrenamiento— `conf_matrix` cambia y estos cuatro números no se actualizan solos: quedan huérfanos. La línea que faltaría, en vez de los valores fijos:
> ```python
> tn, fp, fn, tp = conf_matrix.ravel()
> ```

📊 Como ente gubernamental, hemos desarrollado un proyecto de deep learning para mejorar la detección de diabetes en la comunidad PIME. El beneficio clave de clasificar correctamente los casos es que permite intervenciones médicas tempranas, lo que reduce significativamente los costos asociados con complicaciones a largo plazo y mejora la calidad de vida de los afectados. 💼 Además, al identificar correctamente a los individuos sanos, optimizamos el uso de recursos médicos, evitando gastos innecesarios en pruebas y tratamientos. 🌟 Este enfoque tecnológico no solo protege la salud de nuestra comunidad, sino que también maximiza la eficiencia del sistema de salud, asegurando que los recursos se dirijan a donde más se necesitan. 💪

---
## Resumen de hallazgos sobre el material

| Hallazgo | Gravedad | Detalle |
|---|---|---|
| Entrenamiento sin escalar features | **Alta** | Medido: recall de diabetes 35,4 % (±25,7) sin escalar vs. 61,7 % (±0,6) con `StandardScaler`, misma arquitectura y tres semillas |
| Ceros imposibles sin tratar (faltantes codificados) | **Alta** | `Insulina` 48,7 % de ceros, `PliegueCutaneo` 29,6 %, entre otras — se cargan como mediciones válidas |
| Matriz de confusión hardcodeada en el cálculo económico | **Alta** | No se deriva de `conf_matrix`; queda desconectada de la corrida real del modelo |
| Tres matrices de confusión distintas en el notebook | Media | `conf_matrix` calculada, `[[126,25],[37,43]]` comentada, y `tn=124,fp=27,fn=33,tp=47` hardcodeada — ninguna verificablemente ligada entre sí |
| Comentario de `learning_rate` contradice el código | Baja | Dice "0.003", el código usa `Adam(learning_rate=0.001)` |